# Loading gravitational-wave data from SFDBs and running the Hough transform

Need to do injection part, first sinusoid, then sinusoid plus fdot, then doppler shift, then add antenna patterns


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrew-l-miller/gwosc/blob/main/create_sfdbs/searching_and_injecting_with_SFDBs_and_FH.ipynb)


In [1]:
from pathlib import Path
import os
import requests
import argparse
import matplotlib.pyplot as plt
import numpy as np
import scipy
import glob
import importlib


In [2]:
import pyhough
importlib.reload(pyhough)
import pyfstat
from read_sfdb import *
from pyhough import pm
from pyhough import hm
from pyhough import physics
from pyhough import gfh
from pyhough.provider_injections import *
from pyhough.signal_simulations import *
from pyhough.funcs_to_inject_into_sfts import *


26-02-15 13:05:05.945 pyfstat INFO    : Running PyFstat version 2.0.2


### what the SDFB header contains

`SFDBheader:
  eof             = 0
  endian          = 1.0
  detector        = 2
  gps_sec         = 1369185280
  gps_nsec        = 0
  tbase           = 1024.0
  firstfrind      = 0
  nsamples        = 2097152
  red             = 128
  typ             = 2
  n_flag          = -1.0
  einstein        = 1.0
  mjdtime         = 60091.05164351852
  nfft            = 7
  wink            = 5
  normd           = 7.62939453125e-06
  normw           = 1.2066189050674438
  frinit          = 0.0
  tsamplu         = 0.000244140625
  deltanu         = 0.0009765625
  vx_eq           = 1.22345e-319
  vy_eq           = 4.015697475586e-312
  vz_eq           = 0.0
  px_eq           = 1.034068651e-314
  py_eq           = 2.068136562e-314
  pz_eq           = 3.1022049663e-314
  n_zeroes        = 0
  sat_howmany     = 0.0
  spare1          = 0.0
  spare2          = 0.0
  spare3          = 0.0
  spare4          = 0.0
  spare5          = 0.0
  spare6          = 0.0
  lavesp          = 16384
  spare8          = 1
  spare9          = 0`

In [3]:
!wget -nc https://dcc.ligo.org/public/0192/T2400058/003/segsH1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt
!wget -nc https://dcc.ligo.org/public/0192/T2400058/003/segsL1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt



File 'segsH1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt' already there; not retrieving.

File 'segsL1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt' already there; not retrieving.



In [4]:
obs_run = 'O4a'
ifo = 'H1'
fname_scisegs_for_sfdbs = ifo+'_'+obs_run+'_sciseg_for_sfdb.txt'
from convert_sciseg_file import load_sciseg_file,time_in_science


if obs_run == 'O4a':
    if ifo == 'H1':
        O4a_scisegs = 'segsH1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt'
    elif ifo == 'L1':
        O4a_scisegs = 'segsL1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt'

sci_times = load_sciseg_file(O4a_scisegs)

## Generic functions to take f(t) and amps and inject into an SFT

In [ ]:

def phase_from_frequency(tt,inject_fs):
    """
    inject_fs : array of instantaneous frequencies [Hz]
    tt        : array of times [s], same length

    returns
    -------
    phase : array of phase [rad]
    """
    inject_fs = np.asarray(inject_fs, dtype=float)
    tt = np.asarray(tt, dtype=float)

    dt = np.hstack((0,np.diff(tt)))
    phase_cycles = np.cumsum(inject_fs * dt)
    phase = np.mod(phase_cycles, 1.0) * 2.0 * np.pi
    return phase


def inject_into_sft(tt,inject_fs,amps,sft,NORM=1/2):

    phase = phase_from_frequency(tt,inject_fs)
    sig = amps * np.exp(1j * phase) ## need to include antenna pattern 
    sft = sft + np.fft.fft(sig) * NORM
    return sft


def tdt2tdb(mjd):
# % TDT2TDB  seconds to add to tdt (terrestrial dymamical time (TAI corrected)) 
# %          to have tdb (barycentric dynamical time)
# %
# %   mjd   mjd value (days)
# %
# %   tdb   seconds to add to the tdt

    JD = mjd + 2400000.5
    g = mod(357.53 + 0.98560028 * (JD - 2451545.0),360) * np.pi/180
    tdb = 0.001658 * np.sin(g) + 0.000014 * np.sin(2*g)
    return tdb


def calc_Hplus_Hcross(eta,psi):
#     eta=simsour.eta; #both in radians
#     psi=simsour.psi*pi/180;
    Hp = np.sqrt(1 / (1 + eta**2)) * (np.cos(2 * psi) - 1j * eta * np.sin(2 * psi));
    Hc = np.sqrt(1 / (1 + eta**2)) * (np.sin(2 * psi) + 1j * eta * np.cos(2 * psi));
    
    return Hp,Hc




## Simulate injections like these examples

Return amplitudes and frequencies

In [ ]:
def simulate_sinusoid(tt,f0,h0):
    freqs = f0 * np.ones((len(tt)))
    amps = h0 * np.ones((len(tt)))
    return amps,freqs


def simulate_sinusoid_with_drift(tt,f0,fdot,h0):
    freqs = f0 + fdot * tt
    amps = h0 * (freqs / f0)**2
    return amps,freqs

def simulate_cw(tt,f0,fdot,alpha,delta,h0,vs):
    
    _, freqs_no_dopp = simulate_sinusoid_with_drift(tt, f0, fdot, h0=0.0)  # dummy h0
    vec_n = pyhough.pm.astro2rect([alpha,delta],1)
    freqs = freqs_no_dopp * (1 + np.dot(vec_n,vs.T))
    amps = h0 * (freqs_no_dopp / f0)**2
    return amps,freqs

def simulate_power_law(tt,f0,k,n,h0):
    const=k * (n-1) * f0**(n-1)
    
    if n == 1:
        fsss = f0 * np.exp(-k * tt);
    elif n == 11/3:
        fsss = f0 * (1 - const * tt)**(-1. / (n-1));
    else:
        fsss = f0 * (1 + const * tt)**(-1. / (n-1));
    
    if (n < 6.8) & (n != 11/3):
        amps = h0 * fsss**2 / f0**2 # ns
    elif (n >= 6.8) & (n < 7.2):
        amps = h0 * fsss**3 / f0**3 #rmode
    elif n == 11/3:
        amps = h0 * fsss**(2/3) / f0**(2/3); #binary

 
    return amps,fsss


def simulate_cbc_pn(tt, m1, m2, t_c, h0, order=3.5):
    """
    tt: absolute time array in GPS seconds (or any consistent absolute time base)
    t_c: coalescence time in same units as tt
    """
    tau = t_c - tt  # seconds-to-merger

    freqs, _, _ = cbc_calc_pn_freq(m1, m2, tau, order=order)

    amps = np.full_like(freqs, h0, dtype=float)  # placeholder amplitude
    return amps, freqs


def cbc_calc_pn_freq(m1, m2, tau, order=3.5):
    """

    Parameters
    ----------
    m1, m2 : float or array-like
        Component masses in solar masses.
    tau : float or array-like
        Seconds to merger (tc - t), same shape/broadcastable with m1, m2.
    order : float
        PN order in {1, 1.5, 2, 2.5, 3, 3.5}.
    consts : Constants
        Physical constants container.

    Returns
    -------
    fgw : ndarray
        PN GW frequency (Hz).
    fgw_0pn : ndarray
        Leading-order (0PN) GW frequency (Hz).
    df : ndarray
        abs(fgw - fgw_0pn) (Hz).
    """
    
    cc = physics.constants()
    c = cc['c']
    G = cc['G']
    Msun = cc['Msun']

    m1 = np.asarray(m1, dtype=float)
    m2 = np.asarray(m2, dtype=float)
    tau = np.asarray(tau, dtype=float)

    m = m1 + m2
    nu = (m1 * m2) / (m ** 2)  # symmetric mass ratio

    # theta = nu*c^3/(5*G*m*Msun) * tau
    theta = nu * c**3 / (5.0 * G * m * Msun) * tau

    C = 0.577215664901533  # Euler-Mascheroni gamma

    PN_0 = 1.0
    PN_1 = 0.0
    PN_1_5 = 0.0
    PN_2 = 0.0
    PN_2_5 = 0.0
    PN_3 = 0.0
    PN_3_5 = 0.0

    # Helper powers
    theta_m1_4 = theta ** (-1.0 / 4.0)
    theta_m3_8 = theta ** (-3.0 / 8.0)
    theta_m1_2 = theta ** (-1.0 / 2.0)
    theta_m5_8 = theta ** (-5.0 / 8.0)
    theta_m3_4 = theta ** (-3.0 / 4.0)
    theta_m7_8 = theta ** (-7.0 / 8.0)

    # MATLAB uses exact == comparisons on doubles; in Python be tolerant
    def _is(x, y, tol=1e-12):
        return abs(x - y) <= tol

    if _is(order, 1.0):
        PN_1 = (743/4042 + (11/48) * nu) * theta_m1_4

    elif _is(order, 1.5):
        PN_1 = (743/4042 + (11/48) * nu) * theta_m1_4
        PN_1_5 = -(1/5) * np.pi * theta_m3_8

    elif _is(order, 2.0):
        PN_1 = (743/4042 + (11/48) * nu) * theta_m1_4
        PN_1_5 = -(1/5) * np.pi * theta_m3_8
        PN_2 = (19583/254016 + (24401/193536) * nu + (31/288) * nu**2) * theta_m1_2

    elif _is(order, 2.5):
        PN_1 = (743/4042 + (11/48) * nu) * theta_m1_4
        PN_1_5 = -(1/5) * np.pi * theta_m3_8
        PN_2 = (19583/254016 + (24401/193536) * nu + (31/288) * nu**2) * theta_m1_2
        PN_2_5 = (-(11891/53760) + (109/1920) * nu) * np.pi * theta_m5_8

    elif _is(order, 3.0):
        PN_1 = (743/4042 + (11/48) * nu) * theta_m1_4
        PN_1_5 = -(1/5) * np.pi * theta_m3_8
        PN_2 = (19583/254016 + (24401/193536) * nu + (31/288) * nu**2) * theta_m1_2
        PN_2_5 = (-(11891/53760) + (109/1920) * nu) * np.pi * theta_m5_8
        PN_3 = (
            -10052469856691/6008596070400
            + (1/6) * np.pi**2
            + (107/420) * C
            - (107/3360) * np.log(theta/256)
            + (3147553127/780337152 - (451/3072) * np.pi**2) * nu
            - (15211/442368) * nu**2
            + (25565/331776) * nu**3
        ) * theta_m3_4

    elif _is(order, 3.5):
        PN_1 = (743/4042 + (11/48) * nu) * theta_m1_4
        PN_1_5 = -(1/5) * np.pi * theta_m3_8
        PN_2 = (19583/254016 + (24401/193536) * nu + (31/288) * nu**2) * theta_m1_2
        PN_2_5 = (-(11891/53760) + (109/1920) * nu) * np.pi * theta_m5_8
        PN_3 = (
            -10052469856691/6008596070400
            + (1/6) * np.pi**2
            + (107/420) * C
            - (107/3360) * np.log(theta/256)
            + (3147553127/780337152 - (451/3072) * np.pi**2) * nu
            - (15211/442368) * nu**2
            + (25565/331776) * nu**3
        ) * theta_m3_4
        PN_3_5 = (
            -(113868647/433520640)
            - (31821/143360) * nu
            + (294941/3870720) * nu**2
        ) * np.pi * theta_m7_8

    else:
        raise ValueError("order must be one of {1, 1.5, 2, 2.5, 3, 3.5}")

    # x = 1/4 * theta^(-1/4) * (sum PN terms)
    x = 0.25 * theta_m1_4 * (PN_0 + PN_1 + PN_1_5 + PN_2 + PN_2_5 + PN_3 + PN_3_5)

    omegaS = (c**3) / (G * m * Msun) * (x ** (3.0 / 2.0))
    omega_gw = 2.0 * omegaS
    fgw = omega_gw / (2.0 * np.pi)

    # 0PN (leading order)
    x_0pn = 0.25 * theta_m1_4
    omegaS_0pn = (c**3) / (G * m * Msun) * (x_0pn ** (3.0 / 2.0))
    omega_gw_0pn = 2.0 * omegaS_0pn
    fgw_0pn = omega_gw_0pn / (2.0 * np.pi)

    df = np.abs(fgw - fgw_0pn)
    return fgw, fgw_0pn, df


def calc_k( mc ):
    cc = physics.constants()
    
    c = cc['c']
    G = cc['G']
    msun = cc['Msun']
    
    k=96/5*np.pi**(8/3)*(G*mc*msun/c**3)**(5/3);
    return k



In [ ]:
from dataclasses import dataclass

@dataclass
class InjContext:
    vs: np.ndarray  # vs[fft_index] used by CW only


def provider_sinusoid(f0, h0):
    def inj_provider(tt, fft_index, ctx):
        return simulate_sinusoid(tt, f0, h0)
    return inj_provider


def provider_sinusoid_drift(f0, fdot, h0):
    def inj_provider(tt, fft_index, ctx):
        return simulate_sinusoid_with_drift(tt, f0, fdot, h0)
    return inj_provider


def provider_cw(f0, fdot, alpha, delta, h0):
    def inj_provider(tt, fft_index, ctx):
        return simulate_cw(tt, f0, fdot, alpha, delta, h0, ctx.vs[fft_index])
    return inj_provider


def provider_power_law(f0, k, n, h0):
    def inj_provider(tt, fft_index, ctx):
        return simulate_power_law(tt, f0, k, n, h0)
    return inj_provider

def provider_cbc(m1,m2,t_c,h0):
    def inj_provider(tt,fft_index,ctx=None):
        return simulate_cbc_pn(tt, m1, m2, t_c, h0, order=3.5)


In [ ]:


def calc_fdot_chirp(mc, fgw):
    """
    Spin-up of a chirping gravitational-wave signal (leading PN order).

    Parameters
    ----------
    mc : float or array-like
        Chirp mass in solar masses.
    fgw : float or array-like
        Gravitational-wave frequency in Hz.

    Returns
    -------
    fdot : ndarray
        Frequency derivative (Hz/s).
    """

    consts = physics.constants()
    G = consts['G']
    c = consts['c']
    Msun = consts['Msun']

    mc = np.asarray(mc, dtype=float)
    fgw = np.asarray(fgw, dtype=float)

    # convert chirp mass to kg
    mc_si = mc * Msun

    fdot = (
        (96.0 / 5.0)
        * np.pi**(8.0 / 3.0)
        * (G * mc_si / c**3)**(5.0 / 3.0)
        * fgw**(11.0 / 3.0)
    )

    return fdot

def calc_time_to_coalescence(Mc, fgw):
    """
    Leading-order time to coalescence for a binary inspiral.

    Parameters
    ----------
    Mc : float or array-like
        Chirp mass in solar masses.
    fgw : float or array-like
        Gravitational-wave frequency in Hz.

    Returns
    -------
    tau : ndarray
        Time to coalescence (seconds).
    """

    consts = physics.constants()
    G = consts['G']
    c = consts['c']
    Msun = consts['Msun']

    Mc = np.asarray(Mc, dtype=float)
    fgw = np.asarray(fgw, dtype=float)

    # convert chirp mass to SI (kg)
    Mc_si = Mc * Msun

    tau = (
        (5.0 / 256.0)
        * (np.pi * fgw)**(-8.0 / 3.0)
        * (G * Mc_si / c**3)**(-5.0 / 3.0)
    )

    return tau


def gps2mjd(tgps):
    """
    Convert GPS time (seconds) to Modified Julian Date (days).

    Parameters
    ----------
    tgps : float or array-like
        GPS time in seconds.

    Returns
    -------
    mjd : ndarray
        Modified Julian Date (days).
    """

    tgps = np.asarray(tgps, dtype=float)

    t0 = 44244.0  # MJD at GPS epoch (6-Jan-1980 00:00:00)

    mjd = tgps / 86400.0 + t0

    # Leap second correction (GPS linked to TAI, offset from UTC)
    mjd = mjd - (leap_seconds(mjd) - 19.0) / 86400.0

    return mjd

import numpy as np

# MJD effective dates when TAI-UTC stepped by +1s (from your MATLAB table)
_LEAP_MJD = np.array([
    41317, 41499, 41683, 42048, 42413, 42778, 43144, 43509, 43874,
    44786, 45151, 45516, 46247, 47161, 47892, 48257, 48804, 49169,
    49534, 50083, 50630, 51179, 53736, 54832, 56109, 57204, 57754
], dtype=float)

# nls at/after the last entry in the table above (2017-01-01): TAI-UTC = 37 s
# If new leap seconds occur, you must update both _LEAP_MJD and this value.
_LEAP_MAX = 37


def leap_seconds(mjd):
    """
    Number of leap seconds (TAI-UTC in seconds) applicable at a given MJD.

    Parameters
    ----------
    mjd : float or array-like
        Modified Julian Date (days).

    Returns
    -------
    nls : float or ndarray
        TAI-UTC (seconds). Same shape as input.
    """
    mjd = np.asarray(mjd, dtype=float)

    # Count how many leap dates are strictly less than mjd
    # (MATLAB code uses: if mjd > leaptimes(i) then break)
    n_before = np.searchsorted(_LEAP_MJD, mjd, side="right")

    # At mjd beyond the last leap date, n_before == len(_LEAP_MJD) -> returns _LEAP_MAX
    # At earlier mjd, subtract how many steps haven't happened yet.
    nls = _LEAP_MAX - (len(_LEAP_MJD) - n_before)

    # Return scalar if scalar input
    if nls.shape == ():
        return float(nls)
    return nls


## tCW portion

In [ ]:

import numpy as np
from scipy.signal import medfilt


def pswindow(typ, length, par=None):
    """
    Computes windows for power spectrum estimates.
    
    Args:
        typ : int or str
            Window type:
            0 or 'no'       : no window (flat)
            1 or 'bartlett' : Bartlett window
            2 or 'hanning'  : Hanning window
            3 or 'flatcos'  : flat-top with cosine edge
            4 or 'tukey'    : Tukey window
            5 or 'gauss'    : Gaussian (3 sigma)
        length : int
            Length of the window
        par : float, optional
            Parameter for Tukey window (alpha)
    
    Returns:
        y : np.ndarray
            Window array of shape (length,)
    """
    y = np.ones(length)
    len2 = length // 2
    len4 = int(np.ceil(length / 4))

    # Map numeric types to strings
    if isinstance(typ, (int, float)):
        if typ == 1:
            typ = 'bartlett'
        elif typ == 2:
            typ = 'hanning'
        elif typ == 3:
            typ = 'flatcos'
        elif typ == 4:
            typ = 'tukey'
        elif typ == 5:
            typ = 'gauss'
        else:
            typ = 'no'

    typ = str(typ).lower()

    if typ == 'bartlett':
        y[:len2] = np.arange(1, len2+1) / len2
        y[len2:] = y[len2-1::-1]
        y *= np.sqrt(3)

    elif typ == 'hanning':
        y[:len2] = 1 - np.cos(np.arange(1, len2+1) * np.pi / len2)
        y[len2:] = y[len2-1::-1]
        y *= np.sqrt(2/3)

    elif typ == 'flatcos':
        y[:len4] = (1 - np.cos(np.arange(1, len4+1) * np.pi / len4)) / 2
        y[-len4:] = y[:len4][::-1]
        y *= np.sqrt(length / np.sum(y**2))

    elif typ == 'tukey':
        if par is None:
            par = 0.5  # default alpha if not provided
        lenx = int(round(par * length / 2))
        y[:lenx] = (1 - np.cos(np.arange(1, lenx+1) * np.pi / lenx)) / 2
        y[-lenx:] = y[:lenx][::-1]
        y *= np.sqrt(length / np.sum(y**2))

    elif typ == 'gauss':
        dx = 6 / (length - 1)
        x = np.arange(-3, 3 + dx, dx)
        y = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)
        y *= np.sqrt(length / np.sum(y**2))

    elif typ == 'no':
        y = np.ones(length)

    else:
        raise ValueError(f"Unknown window type: {typ}")

    return y


def get_original_strains(sft):
    opposite = np.flipud(sft);

    appending2 = np.hstack((0, np.conj(opposite[0:-1])))

    old_sft = np.hstack((sft, appending2))

    strains=np.fft.ifft(old_sft)
    
    return strains

def change_FFT_length(sft,sfdb_head,TFFT,minf,maxf,do_inj=False,num_orig_FFT=0,downsamp=False,band=False,win=3):
    
    strains =  get_original_strains(sft)
    
    nsamps = len(strains)
    dt = sfdb_head.tsamplu
    tfft_orig = sfdb_head.tbase
    red = sfdb_head.red
    

    nfft = int(TFFT / dt)
    num1 = nsamps // 4
    num2 = 3 * num1
    strains_to_fft = strains[num1:num2]

    n1 = len(strains_to_fft)
    nover = nfft // 2

#     normd = np.sqrt(dt / nfft)
#     normw = np.sqrt(3 / 2)  # ≈1.2066

    df = 1.0 / TFFT
    Fs = 1.0 / dt
    w = pswindow(win, nfft)  
#     times = np.arange(n1) * dt
    freqs = np.arange(0, Fs/2, df)
    
    ind = 0
    num_FFTs = int(np.ceil(2 * n1 / nfft))
    all_t0s = []

    if (2 * n1) % nfft == 0:
        cond2 = num_FFTs + 10000
    else:
        cond2 = num_FFTs - 2

    k1,k2,fr1,fr2,kss1,kss2,_,_ = get_sft_sps_and_f_inds(minf,maxf,nfft,df,red,1)
        
    print(k1,k2,nfft)
    all_FFTs = np.zeros((num_FFTs, nfft // 2), dtype=complex)
    all_SPSs = np.zeros_like(all_FFTs, dtype=float)
    tf_map = np.zeros_like(all_FFTs, dtype=float)
    gps0 = sfdb_head.gps_sec
    for jj in range(num_FFTs):
        if ind == 0:
            x = strains_to_fft[:nfft]
        elif ind == num_FFTs - 1 or ind == cond2:
            x = strains_to_fft[ind*nover:]
            if (2*n1) % nfft == 0:
                x = np.concatenate((x, np.zeros(nfft//2)))
            else:
                x = np.concatenate((x, np.zeros(nfft - len(x))))
        else:
            x = strains_to_fft[ind*nover : ind*nover + nfft]

        x = x * w
        xx = np.fft.fft(x)
        full_sft = np.sqrt(2) * xx[:nfft//2] ### sqrt(2) is bilat to unilat parameter, coming from neglecting neg freqs
        new_gps_sec = gps0 + (1/2) * ind * TFFT
        if do_inj == True:
            if downsamp:
                dsfact,NORM = calc_dsfact(dt,minf,maxf)
#                 k1,k2,fr1,_,kss1,kss2,_,_ = get_sft_sps_and_f_inds(minf,maxf,nfft,df,red,dsfact)
                nsamp_orig = len(full_sft)
                dtnew,nfftnew  = get_downsampled_times_samps( dt,nsamp_orig,dsfact,full_sft[k1:k2] )
                inj_times = np.arange(nfftnew) * dtnew
                
                if jj == 0:
                    all_FFTs = np.zeros((num_FFTs, nfftnew), dtype=complex)
                    all_SPSs = np.zeros_like(all_FFTs, dtype=float)
                    tf_map = np.zeros_like(all_FFTs, dtype=float)
                
            else:
#                 k1,k2,fr1,fr2,kss1,kss2,_,_ = get_sft_sps_and_f_inds(minf,maxf,nfft,df,red,1)
#                 k1 = 0
#                 k2 = nfft-1
#                 fr1 = 0
                NORM = 1.0 / 2.0
                if band:
                    nfftnew = k2 - k1
                    if jj == 0:
                        all_FFTs = np.zeros((num_FFTs, nfftnew), dtype=complex)
                        all_SPSs = np.zeros_like(all_FFTs, dtype=float)
                        tf_map = np.zeros_like(all_FFTs, dtype=float)
                    
                    inj_times = np.arange(nfftnew) * dt * 2
                else:
                    k1 = 0
                    k2 = nfft-1
                    inj_times = np.arange(nfft / 2) * dt * 2 ### nfft/2 b/c nfft is including neg freq samples                
                
            tt = inj_times + (new_gps_sec - gps0) + num_orig_FFT * tfft_orig / 2

            print("fft index:",fft_index)
            print("num_orig_FFTnum_orig_FFT)
            amps, fsss = inj_provider(tt, fft_index,ctx=None)
            if downsamp:
                fsss = fsss - fr1
            
            if np.sum(np.abs(full_sft)) == 0:
                full_sps = full_sft[k1:k2] * 0
                full_sft = full_sps
            else:
                spec_freqs = freqs[k1:k2]
#                 plt.semilogy(spec_freqs,np.abs(full_sft[k1:k2]))
#                 plt.show()
                full_sft = inject_into_sft(tt,fsss,amps,full_sft[k1:k2],NORM)
                full_sps = np.sqrt(medfilt(np.abs(full_sft)**2, kernel_size=21)) / np.log(2)
        if band:
            full_sft = full_sft[k1:k2]
        # PSD estimate with median filter
        
            if jj == 0:
                all_FFTs = np.zeros((num_FFTs, 2*len(full_sft) // 2), dtype=complex)
                all_SPSs = np.zeros_like(all_FFTs, dtype=float)
                tf_map = np.zeros_like(all_FFTs, dtype=float)
        
        full_sps = np.sqrt(medfilt(np.abs(full_sft)**2, kernel_size=21)) / np.log(2)
        all_FFTs[jj, :] = full_sft
        all_SPSs[jj, :] = full_sps
        tf_map[jj, :] = np.abs(full_sft)**2 / full_sps**2

        all_t0s.append(new_gps_sec)
        ind += 1

    return all_t0s,freqs,all_FFTs.T, all_SPSs.T, tf_map.T



def sub_whitenoise(lfft, header,ampnoise=7.94e-24):
    """
    Generate complex white noise SFT (column vector style).

    Parameters
    ----------
    lfft : int
        FFT length
    ampnoise : float
        Noise amplitude scaling
    header : object
        Must have attributes:
            header.normd
            header.normw

    Returns
    -------
    sft : ndarray, shape (lfft,)
        Complex white noise vector
    """

    scale = (
        (1.0 / header.normd) *
        (1.0 / header.normw) *
        ampnoise *
        (1.0 / np.sqrt(2.0))
    )

    sft = (np.random.randn(lfft) + 1j * np.random.randn(lfft)) * scale

    median_abs = np.median(np.abs(sft))

    sps = (median_abs + sft * 0.0) \
          * header.normd \
          * header.normw \
          * np.sqrt(2.0)

    sps = sps * np.sqrt(1.0 / np.log(2.0))

    return sft,sps

import numpy as np


def calc_dsfact(dt, minf,maxf):
    """
    Compute downsampling factor and normalization.

    Parameters
    ----------
    dt : float
        Original time sampling interval.
    minf : array-like
    maxf : array-like
        min and max freqeuncies of the band to downsample to

    Returns
    -------
    dsfact : float
        Downsampling factor.
    NORM : float
        FFT normalization factor.
    """

    f_band_you_want = maxf - minf
    dt_prime = 1.0 / f_band_you_want
    dsfact = dt_prime / dt

    if dsfact > 1:
        # normalization for FFT because of downsampling
        # factor 2 due to complex vs real convention (PIA 12 Oct 2016)
        NORM = np.sqrt(dsfact * dsfact) / 2.0
        # equivalently: NORM = dsfact / 2.0
    else:
        NORM = 1.0 / 2.0

    return dsfact, NORM


def get_downsampled_times_samps(dtori,nsamples, dsfact, sft):
    """
    Compute new sampling time and FFT length after downsampling.

    Parameters
    ----------
    header : object or dict
        Must contain:
            - tsamplu   (original time sampling)
            - nsamples  (number of stored samples)
    dsfact : float
        Downsampling factor
    sft : array-like
        SFT data array

    Returns
    -------
    dtnew : float
        New sampling time
    lfftnew : int
        New FFT length
    """

    lfftori = nsamples * 2  # times 2 because extends to negative freqs

    if dsfact > 1:
        dtnew = dtori * dsfact
        lfftnew = lfftori / dsfact

        # Ensure lfftnew is integer-consistent with sft length
        if not np.isclose(np.floor(lfftnew), lfftnew):
            lfftnew_int = int(np.floor(lfftnew))
            lala = len(dtnew * np.arange(lfftnew_int))

            if lala < len(sft):
                lfftnew = int(np.ceil(lfftnew))
            elif lala > len(sft):
                lfftnew = int(np.floor(lfftnew))
            else:
                lfftnew = lfftnew_int
        else:
            lfftnew = int(lfftnew)

    else:
        lfftnew = lfftori
        dtnew = dtori

    return dtnew, lfftnew


def get_sft_sps_and_f_inds(minf,maxf, nsamp,dfr, red,dsfact):
    """
    Translate MATLAB function to Python with 0-based indexing.
    
    Parameters:
    -----------
    freq : array-like
        Frequency range [freq_min, freq_max]
    piahead : object
        Object with attributes: nsamples, deltanu, firstfrind, red
    dsfact : int
        Downsampling factor
        
    Returns:
    --------
    k1, k2, fr1, fr2, kss1, kss2, k1int, k2int : int or float
        Various frequency indices and values
    """
#     nsamp = piahead.nsamples
#     dfr = piahead.deltanu
    inifr0 = 0 * dfr
    finfr0 = (nsamp - 1) * dfr + inifr0
    
    # Convert from MATLAB 1-based to Python 0-based indexing
    k1 = int(np.floor(minf / dfr + 0.0001))
    fr1 = k1 * dfr
    
    k2 = int(np.round(maxf / dfr)) - 1  # Subtract 1 for 0-based indexing
    
    if dsfact > 1:
        if k2 > nsamp - 1:  # Adjust for 0-based indexing
            k2 = nsamp - 1
            if np.floor((k2 - k1) / 2) * 2 == k2 - k1:
                k2 = k2 - 1
    
    fr2 = k2 * dfr
    
    frss1 = max(fr1, inifr0)
    kss1 = int(np.round(frss1 / (dfr * red)))
    
    frss2 = min(fr2, finfr0)
    kss2 = int(np.round(frss2 / (dfr * red)))
    
    var1 = kss2 - kss1 + 1
    var2 = k2 - k1 + 1
    num = var2 / var1
    
    if var1 != var2 and np.floor(num) != num:
        kss1 = kss1 + 1
        var1 = kss2 - kss1 + 1
        num = var2 / var1
        if np.floor(num) != num:
            kss1 = kss1 + 1
    
    if dsfact > 1:
        f0int = np.floor(fr1)
        k1int = int((fr1 - f0int) / dfr)  # Adjust for 0-based indexing
        k2int = int(np.round((fr2 - f0int) / dfr))  # Adjust for 0-based indexing
    else:
        k1int = k1
        k2int = k2
    
     # Adjust k2 and k2int for Python's exclusive-end slicing convention
    k2 = k2 + 1
    k2int = k2int + 1
    return k1, k2, fr1, fr2, kss1, kss2, k1int, k2int

## tCW main injection script

In [ ]:
sfdb_files = glob.glob("/Users/andrewmiller/Downloads/*.SFDB09")

if not sfdb_files:
    raise FileNotFoundError("No .SFDB09 files found in the current directory.")
else:
    sfdb_file = sfdb_files[0]
    print(f"Using SFDB file: {sfdb_file}")
minf = 800.0
maxf = 887.0

allt0s = []
fft_index = 0

max_num_ffts = 100

do_inj = True
white_noise = True
downsamp = False
band = False

mc = 5.9e-4
n = 11/3

fdotmin = calc_fdot_chirp(mc,minf); # calculate minimum fdot
fdotmax = calc_fdot_chirp(mc,maxf); # calculate maximum fdot
t1 = calc_time_to_coalescence(mc,minf); # time left to coalesence at minf
t2 = calc_time_to_coalescence(mc,maxf); # time left to coalesence at maxf
dur = np.floor(t1-t2); # duration analyzed
new_tfft = np.round(1/np.sqrt(fdotmax)) # confine all frequency modulations to 1 freq bin in each FFT

# inj_provider = provider_sinusoid_drift(f0=860, fdot=1e-3, h0=1e-23)

inj_provider = provider_power_law(f0=800,k=calc_k(mc),n=n,h0=2.81e-23)

print("duration (s): ", dur)
print("TFFT (s): ",new_tfft)

with open(sfdb_file, "rb") as f:
    while True:
        sfdb_head, _, sps, sft = sfdb_read_an_FFT(f)
        if sfdb_head == 0:
            break
        if (np.sum(sps) == 0) or (np.sum(np.abs(sft)) == 0):
            continue
        if fft_index == 0:
            t0 = sfdb_head.gps_sec
            dt = sfdb_head.tsamplu
            tfft = sfdb_head.tbase
            nsam = sfdb_head.nsamples
            df = sfdb_head.deltanu
            N_FFT = np.ceil(dur / (tfft / 2))
            t_fin = t0 + dur
            _,_,t_in,_ = time_in_science(t0,t_fin,sci_times)

            if white_noise == False:
                if t_in / dur < 0.99:
                    continue
                else:
                    print('signal will be completely in sci time')
    
            
            norm_factor = sfdb_head.normd * sfdb_head.normw * np.sqrt(2)
            all_t0s_in_sfdb = np.arange(max_num_ffts)*tfft/2+t0
#             vs = get_detector_velocities(all_t0s_in_sfdb,tfft,ifo)
        
        if white_noise:
            lfft = int(tfft / dt)
            sft,sps = sub_whitenoise(lfft,sfdb_head)
            
        times,freqs,FFTs, SPSs, tf_map = change_FFT_length(sft,sfdb_head,new_tfft,minf,maxf,do_inj,fft_index,downsamp,band)

        if fft_index == 0:
            whole_map = tf_map.copy()
        else:
            whole_map = np.concatenate((whole_map, tf_map), axis=1)

        allt0s.extend(times)
        
        print(f"[{fft_index+1}/{int(N_FFT+1)}] FFTs complete.")

        fft_index += 1
        if fft_index > N_FFT:
            cut_inds = (allt0s <= t_fin)
            allt0s = np.asarray(allt0s)[cut_inds]
            whole_map = whole_map[:,cut_inds]
            break

        



In [ ]:
xv = whole_map[(freqs>minf) & (freqs<maxf),:].flatten()
plt.hist(
    xv,
    bins=200,
    density=True,
    histtype='step',
    lw=2.0,
    color='tab:green',
    label='data'
)

# Theoretical exponential PDF (mean=1)
xx = np.linspace(0, np.percentile(xv, 99.9), 2000)  # just for drawing the curve nicely
plt.plot(
    xx,
    np.exp(-xx),
    'r--',
    lw=2.2,
    label=r'$\exp(-x)$'
)

plt.xlabel(r'$|{\rm FFT}|^2 / {\rm PSD}$', fontsize=14)
plt.ylabel('Probability density', fontsize=14)

plt.grid(True, which='major', alpha=0.35)
plt.grid(True, which='minor', alpha=0.15)
plt.minorticks_on()

# No tail clipping of the data, but you *do* want sane axes for visibility:
plt.xlim(0, np.percentile(xv, 99.5))
plt.ylim(bottom=0)

plt.legend(frameon=True, fontsize=12, loc='upper right')

plt.tight_layout()
plt.show()

print("mean:", np.mean(xv))
print("std:", np.std(xv))
print("median:", np.median(xv))

In [ ]:


threshold = 2.5
pm_times,pm_freqs,pm_pows,index = pyhough.pm.make_peakmap_from_spectrogram(allt0s,freqs,whole_map,threshold)

In [ ]:
# in_band = (pm_freqs > minf-minf) & (pm_freqs<maxf-minf)


pyhough.pm.python_plot_triplets((pm_times-pm_times[0]),pm_freqs,pm_pows,'.',label='equalized power');
plt.xlabel('time (s)',size=14)
plt.ylabel(r'frequency (Hz)',size=14);

In [ ]:
in_band = (pm_freqs > minf) & (pm_freqs<maxf)


pyhough.pm.python_plot_triplets((pm_times[in_band]-pm_times[0]),pm_freqs[in_band],(pm_pows[in_band]),'.',label='equalized power');
plt.xlabel('time (s)',size=14)
plt.ylabel(r'frequency (Hz)',size=14);

# peaks_in_x = 1 / pm_freqs[in_band]**(n-1) # Eq. 3
# pyhough.pm.python_plot_triplets((pm_times[in_band]-pm_times[0]),peaks_in_x,pm_pows[in_band],'.',label='equalized power')
# plt.xlabel('time (s)',size=14)
# plt.ylabel(r'x (Hz$^{-8/3}$)',size=14);

In [ ]:
ref_perc_time = 0.5
gridk,dk = pyhough.gfh.andrew_long_transient_grid_k(new_tfft,[minf, maxf],[fdotmin, fdotmax],dur,n);
gridk = np.squeeze(gridk)
# gridk = np.squeeze(pyhough.gfh.cbc_shorten_gridk(gridk,sour['kn'],sour['kn']))

t00_ref_time = allt0s[0] + dur * ref_perc_time;
epoch = gps2mjd(t00_ref_time);

hm_job = pyhough.gfh.make_hm_job_struct(minf,maxf,new_tfft,dur,n,ref_perc_time,gridk,epoch)

In [ ]:

p = np.array([
    gps2mjd(pm_times[in_band]),
    pm_freqs[in_band],
    pm_pows[in_band]
])

# hmap,info = pyhough.gfh.hfdf_hough_transients(p, hm_job)
hmap,info = LongT_GENERALIZED_fasthough_nonuni(p,hm_job)

## Plot Hough map in physical coordinates

In [ ]:
# fig, ax = plt.subplots()
# ax.set(ylabel="$k$ (Hz$^{-5/3}$)", xlabel=r"$x_0$ [Hz$^{-8/3}$]")
# c = ax.pcolormesh(
#     info['gridx'],
#     np.squeeze(info['gridk']),
#     hmap,
#     cmap="inferno",
#     shading="nearest",
# )
# fig.colorbar(c, label="number count")
# plt.tight_layout()
# # plt.ylim([2.6e-10,2.8e-10])
# # plt.xlim([1.1e-6,1.3e-6])

mcsss = pyhough.physics.calc_mc_with_k(info['gridk'])
fffss = pyhough.gfh.get_f0_from_x0(info['gridx'],n)
fig, ax = plt.subplots()#figsize=(0.8 * 16, 0.8 * 9))
ax.set(ylabel=r"$\mathcal{M}$ $[M_\odot]$", xlabel=r"frequency [Hz]")
c = ax.pcolormesh(
    fffss,
    np.squeeze(mcsss),
    hmap,
    cmap="inferno",
    shading="nearest",
)
fig.colorbar(c, label="number count")
plt.yscale('log')
plt.tight_layout()
# plt.ylim(4.7e-4,5.5e-4)
# plt.savefig('hm_pbh_inj.png',dpi=400)

In [ ]:
import numpy as np

def LongT_GENERALIZED_fasthough_nonuni(peakss, hm_job):
    """
    Creates a x/k Hough map with non-uniform x-grid binning
    
    This code takes as input a peakmap and first transforms t/f --> t/x
    according to the braking index, then maps t/x --> x/k using the Hough
    
    Parameters:
    -----------
    peakss : ndarray
        peaks(3,n) - peaks of the peakmap as [t,fr,amp] 
        (fr corrected for the Doppler effect)
        Row 0: time (MJD)
        Row 1: frequency (Hz)
        Row 2: amplitude (not used)
    
    hm_job : dict
        Hough map structure containing:
            'minf' : minimum frequency of Hough map (Hz)
            'maxf' : maximum frequency of Hough map (Hz)
            'df' : frequency resolution (1/TFFT) (Hz)
            'dur' : duration of peakmap (s)
            'patch' : [Longitude Latitude] (ecliptic)
            'n' : braking index
            'ref_perc_time' : percentile of reference time [0,1]
            'frenh' : frequency enhancement (1)
            'gridk' : grid on constant k parameter
            'epoch' : reference time (MJD)
    
    Returns:
    --------
    hfdf : ndarray
        Hough map (transposed histogram)
    hm_job : dict
        Updated hough map structure with additional fields:
            'gridx' : x-grid values
            'dx' : spacing in x grid (Hz^{1-n})
            'which_hough' : 'gfh_nonuni'
    """
    
    Day_inSeconds = 86400
    
    gridk = hm_job['gridk'].copy()
    braking_index = hm_job['n']
    
    # Flip spindowns to spinups for certain braking indices
    if braking_index not in [5, 3, 7]:
        # disp('chirp, flipping spindowns to spinups')
        gridk = -gridk
    
    pow_val = braking_index - 1
    
    n2 = peakss.shape[1]
    weights = np.ones(n2)
    
    epoch = hm_job['epoch']
    tpeaks = Day_inSeconds * (peakss[0, :] - epoch)
    
    minf0 = hm_job['minf']
    maxf0 = hm_job['maxf']
    df = hm_job['df']
    enh = hm_job['frenh']
    
    if braking_index == 1:  # case of pulsar winds
        xpeaks = np.log(peakss[1, :])
        pow_val = 1  # not physical, negates pow in each expression
    else:
        xpeaks = peakss[1, :] ** (-pow_val)
    
    # Create non-uniform grid
    freq_grid = np.arange(minf0, maxf0 + df, df)
    gridx = np.flip(1.0 / (freq_grid ** pow_val))
    
    # Call the fast vectorized version
    binh_df0 = original_version_nonuni_fast(
        xpeaks, tpeaks, gridk, gridx, weights, braking_index
    )
    
    hfdf = binh_df0
    
    # Update hm_job with output parameters
    hm_job['gridx'] = gridx[:-1]
    hm_job['dx'] = np.diff(gridx)
    hm_job['which_hough'] = 'gfh_nonuni'
    
    return hfdf, hm_job


def original_version_nonuni_fast(xpeaks, tpeaks, gridk, gridx, weights, braking_index):
    """
    Fully vectorized fast non-uniform x-grid binning using list comprehension
    Inspired by LongT_GENERALIZED_fasthough vectorization
    
    Parameters:
    -----------
    xpeaks : array_like
        Transformed peak frequencies (x = f^(-pow))
    tpeaks : array_like
        Peak times in seconds (relative to epoch)
    gridk : array_like
        Grid of k parameter values to search
    gridx : array_like
        Non-uniform x-grid edges
    weights : array_like
        Weight for each peak (usually all 1s)
    braking_index : float
        Braking index
    
    Returns:
    --------
    binh_df0_orig : ndarray
        Hough map of shape (nbin_k, num_bins)
    """
    
    pow_val = braking_index - 1
    num_bins = len(gridx) - 1
    
    # Create bin edges from the grid points
    bin_edges = np.concatenate([gridx, [np.inf]])
    
    # Normalize time by smallest k step for numerical stability
    dk_diff = np.diff(gridk)
    if len(dk_diff) > 0:
        dk = np.min(np.abs(dk_diff))
    else:
        dk = 1.0
    
    slopes = gridk / dk
    tt_norm = tpeaks * dk * pow_val
    
    # THE KEY VECTORIZATION: Use list comprehension to process all k values at once
    # This is the same trick as LongT_GENERALIZED_fasthough
    hfdf_list = [
        discretize_and_accumulate(xpeaks, tt_norm, slope, bin_edges, num_bins, weights)
        for slope in slopes
    ]
    
    # Convert list to matrix (stack as columns, then transpose)
    binh_df0_orig = np.column_stack(hfdf_list).T
    
    return binh_df0_orig


def discretize_and_accumulate(xx, tt_norm, slope, bin_edges, num_bins, weights):
    """
    Compute x0 values for this slope and bin them
    
    Parameters:
    -----------
    xx : array_like
        X values (transformed frequencies)
    tt_norm : array_like
        Normalized time values
    slope : float
        Current k slope value
    bin_edges : array_like
        Bin edges for discretization
    num_bins : int
        Number of bins
    weights : array_like
        Weight for each peak
    
    Returns:
    --------
    counts : ndarray
        Histogram counts for this slope
    """
    # Compute x0 values for this slope
    x0s = xx - tt_norm * slope
    
    # Discretize into bins
    # np.digitize returns 1-based indices, subtract 1 for 0-based
    bin_idx = np.digitize(x0s, bin_edges) - 1
    
    # Filter valid bins
    valid = (bin_idx >= 0) & (bin_idx < num_bins)
    
    # Accumulate with weights
    if np.any(valid):
        counts = np.bincount(
            bin_idx[valid], 
            weights=weights[valid], 
            minlength=num_bins
        )[:num_bins]
    else:
        counts = np.zeros(num_bins)
    
    return counts

In [ ]:
!pwd